# Лабораторная работа 2. Эмпирический риск и метод наименьших квадратов

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| Место в курсе | после лекции 1 |
| Опора на лекции | лекция 1: функция потерь (опр. 1.11), эмпирический и истинный риск (опр. 1.12–1.13), утв. 1.14 о несмещённости, принцип ERM (опр. 1.15), §1.4 (матричное дифференцирование), нормальные уравнения МНК (теорема 1.24), примеры 1.4 и 1.8 |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Воспроизвести численный пример МНК из конспекта — и убедиться, что утверждения лекции 1 не формальность: эмпирический риск действительно несмещённо оценивает истинный, но **только** для алгоритма, выбранного независимо от выборки. Разобрать, как выбор функции потерь определяет, что именно мы восстанавливаем, и научиться измерять качество регрессии.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab02_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Данные занятия — учебные, одинаковые у всех.** Числа на экране у преподавателя
> и у вас совпадают, поэтому можно сверяться с соседом и спорить о результате вслух.
> Индивидуальная таблица, порождённая по вашему ФИО, появится в домашней работе.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.linear_model import LinearRegression

---
# Часть 1. Воспроизводим пример из конспекта

Начинаем не с теории, а с **эталона** — задачи, ответ которой известен заранее.
Приём простой и работает весь курс: если реализация не даёт известного ответа,
проверять на ней что-то ещё бессмысленно. Эталон не обязан быть большим — он
обязан быть проверяемым: здесь это восемь объектов и три параметра.

Теорема 1.24: если $X^{\mathsf T}X$ невырождена, то
$\theta^* = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y$.

Формулу с обратной матрицей **не реализуют буквально**. Обращать матрицу, чтобы
тут же умножить её на вектор, — лишний шаг, а каждый лишний шаг стоит верных
знаков: `np.linalg.solve` идёт к ответу сразу, `np.linalg.lstsq` — через SVD, ещё
устойчивее. Отсюда правило, которым мы будем пользоваться весь курс:
**`solve` и `lstsq` вместо `inv`.** Насколько это принципиально, вы измерите в
домашней работе на матрице Вандермонда.

Пример 1.4 конспекта описывает ровно эту задачу — восемь кошек, $f_1$ — банок
корма в день, $f_2$ — возраст, $y$ — вес, — но **чисел не приводит**: там задана
постановка, а не расчёт. Числа мы задали сами в конце занятия 1. Посчитав
$\theta^*$ один раз и убедившись, что все способы решения дают одно и то же, мы
получаем **собственный эталон**:
$$\theta^*\approx(2.2009,\ 0.8931,\ 0.1136).$$
К нему мы вернёмся в домашней работе, где им проверяются пять разных решателей
МНК, и дальше по курсу — всякий раз, когда нужен ответ, известный точно.

In [ ]:
f1 = np.array([1, 2, 2, 3, 2, 3, 1, 4], dtype=float)      # банок корма в день
f2 = np.array([1, 2, 3, 5, 7, 10, 2, 8], dtype=float)     # возраст
y_cats = np.array([3.0, 4.2, 4.5, 5.5, 4.8, 6.0, 3.4, 6.6])
X_cats = np.column_stack([np.ones(8), f1, f2])            # столбец единиц под theta_0

theta_solve = np.linalg.solve(X_cats.T @ X_cats, X_cats.T @ y_cats)
theta_lstsq = np.linalg.lstsq(X_cats, y_cats, rcond=None)[0]
theta_ref = np.array([2.2009, 0.8931, 0.1136])            # эталон занятия 1

print(f"нормальные уравнения (solve): {np.round(theta_solve, 4)}")
print(f"SVD-решение (lstsq)         : {np.round(theta_lstsq, 4)}")
print(f"эталон занятия 1            : {theta_ref}")
print(f"\nмаксимальное отличие от конспекта: "
      f"{np.abs(theta_solve - theta_ref).max():.2e}  "
      f"{'-- совпало' if np.abs(theta_solve - theta_ref).max() < 1e-3 else '-- РАСХОЖДЕНИЕ'}")

> **Напоминание — число обусловленности.** $\mathrm{cond}(A)$ — отношение наибольшего сингулярного числа $A$ к
> наименьшему. Практический смысл: если $\mathrm{cond}(A)=10^{k}$, то при решении
> системы с матрицей $A$ теряется примерно $k$ верных десятичных знаков из
> имеющихся шестнадцати; у вырожденной матрицы $\mathrm{cond}=\infty$. В NumPy:
> `np.linalg.cond(A)`. Ключевой факт этой части:
> $\mathrm{cond}(X^{\mathsf T}X) = \mathrm{cond}(X)^2$ — переходя к нормальным
> уравнениям, мы **удваиваем** потерю точности.

In [ ]:
print(f"cond(X)     = {np.linalg.cond(X_cats):10.2f}")
print(f"cond(X^T X) = {np.linalg.cond(X_cats.T @ X_cats):10.2f}"
      f"   = cond(X)^2 = {np.linalg.cond(X_cats) ** 2:.2f}")

> **Вывод.** Совпало ли с конспектом? Что означает $\mathrm{cond}(X^{\mathsf T}X) = \mathrm{cond}(X)^2$ для точности вычислений?
>
> *(ваш ответ здесь)*

---
# Часть 2. Функция потерь задаёт, что мы восстанавливаем

Определение 1.11: потеря $\mathcal L(a,x,y)\ge0$ измеряет ошибку на одном объекте;
в регрессии она зависит только от невязки $r = a(x) - y$. Эмпирический риск
(опр. 1.12) — её среднее по выборке:

$$
Q(a, X^\ell) = \frac1\ell\sum_{i=1}^{\ell}\mathcal L(a, x_i, y_i).
$$

Возьмём четыре потери и посмотрим на них как на функции невязки.

> **Напоминание — квантильная потеря и потеря Хьюбера.** **Квантильная** потеря штрафует перепрогноз и недопрогноз *разными* весами:
> $\mathcal L(r) = (1-q)\,r$ при $r\ge0$ и $-q\,r$ при $r<0$. При $q = 0.5$
> это половина абсолютной потери, при $q = 0.9$ недопрогноз в девять раз дороже
> перепрогноза. Нужна там, где ошибки несимметричны по цене: занизить запас
> товара на складе хуже, чем завысить.
>
> **Хьюбера** — компромисс между квадратичной и абсолютной: квадратичная при
> $|r|\le\delta$ и линейная дальше. Гладкая в нуле (в отличие от абсолютной,
> которую неудобно минимизировать градиентными методами) и не даёт выбросам
> раздувать функционал квадратично.

> **Напоминание — NumPy: `np.where` вместо if.** `np.where(условие, a, b)` — поэлементный выбор: там, где условие истинно,
> берётся значение из `a`, иначе из `b`. Это векторизованный аналог `if`, и он
> нужен потому, что обычный `if` работает с одним числом, а не с массивом:
> `if r >= 0` на массиве вызовет ошибку «truth value of an array is ambiguous».
>
> Все три аргумента подчиняются broadcasting, поэтому `a` и `b` могут быть и
> скалярами: `np.where(x > 0, 1.0, -1.0)` даёт массив из единиц и минус единиц
> той же формы, что `x`.

In [ ]:
def loss_squared(r):
    return r ** 2


def loss_absolute(r):
    return np.abs(r)


def loss_quantile(r, q=0.75):
    """Перепрогноз (r > 0) штрафуется весом 1 - q, недопрогноз -- весом q."""
    return np.where(r >= 0, (1.0 - q) * r, -q * r)


def loss_huber(r, delta=1.0):
    """Квадратичная при |r| <= delta, линейная дальше."""
    a = np.abs(r)
    return np.where(a <= delta, 0.5 * r ** 2, delta * (a - 0.5 * delta))

In [ ]:
r = np.linspace(-3, 3, 400)
fig, ax = plt.subplots()
for loss, name in [(loss_squared, "квадратичная"), (loss_absolute, "абсолютная"),
                   (loss_huber, "Хьюбера, delta=1"),
                   (loss_quantile, "квантильная, q=0.75")]:
    ax.plot(r, loss(r), lw=2, label=name)
ax.set_xlabel("невязка $r = a(x) - y$"); ax.set_ylabel(r"$\mathcal{L}(r)$")
ax.set_title("Четыре функции потерь"); ax.legend()
plt.tight_layout(); plt.show()

### Задание 2.1. Эмпирический риск

Простейшая модель алгоритмов — константы $a(x)\equiv c$. Принцип ERM (опр. 1.15)
требует найти $c^* = \arg\min_c Q(c)$. Начнём с самой функции $Q$.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def empirical_risk(loss, pred, y):
    """Q(a, X^l) -- среднее значение потерь по выборке."""
    # TODO (1 строка): среднее значение loss(pred - y)
    raise NotImplementedError


y_sample = rng.gamma(shape=2.0, scale=3.0, size=400)     # заведомо несимметричное
print(f"Q константы 5.0 при квадратичной потере: "
      f"{empirical_risk(loss_squared, 5.0, y_sample):.3f}")

In [ ]:
# Минимизируем Q(c) по константе c для каждой потери
grid = np.linspace(y_sample.min(), y_sample.max(), 600)
optimum = {}
for loss, name in [(loss_squared, "квадратичная"), (loss_absolute, "абсолютная"),
                   (loss_quantile, "квантильная, q=0.75")]:
    risks = np.array([empirical_risk(loss, c, y_sample) for c in grid])
    optimum[name] = grid[risks.argmin()]

for name, c in optimum.items():
    print(f"{name:22s}: c* = {c:6.3f}")
print(f"\nсреднее выборки  = {y_sample.mean():6.3f}")
print(f"медиана          = {np.median(y_sample):6.3f}")
print(f"квантиль 0.75    = {np.quantile(y_sample, 0.75):6.3f}")

> **Вывод.** Какой выборочной характеристике равен $c^*$ для каждой потери? Продифференцируйте $Q(c)$ по $c$ для квадратичной потери и убедитесь.
>
> *(ваш ответ здесь)*

---
# Часть 3. Проверяем утверждение 1.14

Утверждение 1.14: если $X^\ell$ — простая выборка, то
$\mathbb{E}\bigl[Q(a, X^\ell)\bigr] = R(a)$.

Ключевое слово — **алгоритм $a$ фиксирован до того, как увидена выборка**.
В доказательстве используется, что потери $\mathcal L(a, x_i, y_i)$ одинаково
распределены; если $a$ сам подобран по $X^\ell$, это рассуждение рушится.

Проверим оба случая на модели, где всё считается точно. Зададим распределение
$p(x,y)$ **явно**:

$$
x\sim U[0,1],\qquad y = 1 + 2x + \varepsilon,\qquad
\varepsilon\sim\mathcal N(0,\ \sigma^2),\quad \sigma^2 = 0.25 .
$$

Для истинного алгоритма $a^*(x) = 1 + 2x$ невязка равна ровно шуму, поэтому
истинный риск известен точно: $R(a^*) = \mathbb{E}\varepsilon^2 = \sigma^2 = 0.25$.
Это снова численный эталон — теперь для статистического утверждения.

In [ ]:
THETA_TRUE = np.array([1.0, 2.0])          # свободный член и наклон
SIGMA2, ELL, N_REPEAT = 0.25, 20, 10_000


def sample(n, generator):
    """Простая выборка (i.i.d., §1.3) из распределения p(x, y)."""
    x = generator.uniform(0, 1, n)
    X = np.column_stack([np.ones(n), x])
    return X, X @ THETA_TRUE + generator.normal(0, np.sqrt(SIGMA2), n)


gen = np.random.default_rng(RANDOM_STATE)
X_big, y_big = sample(1_000_000, gen)      # «генеральная совокупность»

print(f"R(a*) теоретический = sigma^2      = {SIGMA2:.4f}")
print(f"R(a*) по выборке 10^6 объектов     = "
      f"{np.mean((X_big @ THETA_TRUE - y_big) ** 2):.4f}")

### Задание 3.1. Фиксированный алгоритм

Повторим эксперимент $N = 10^4$ раз: генерируем выборку размера $\ell = 20$ и
считаем $Q(a^*, X^\ell)$ для **истинного** алгоритма — того самого, что был
известен до всех выборок.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

Q_fixed = np.empty(N_REPEAT)
for t in range(N_REPEAT):
    X, y = sample(ELL, gen)
    # TODO (1 строка): Q для ИСТИННОГО алгоритма, то есть предсказания X @ THETA_TRUE
    Q_fixed[t] = ...

print(f"R(a*)      = {SIGMA2:.5f}")
print(f"среднее Q  = {Q_fixed.mean():.5f}")
print(f"смещение   = {Q_fixed.mean() - SIGMA2:+.5f}")

In [ ]:
# Несмещённость -- это про среднее. А как ведёт себя разброс самого Q?
rows = []
for l in (5, 20, 80, 320):
    Q_l = np.array([empirical_risk(loss_squared, X @ THETA_TRUE, y)
                    for X, y in (sample(l, gen) for _ in range(4000))])
    rows.append({"l": l, "среднее Q": Q_l.mean(), "ст. откл. Q": Q_l.std(),
                 "sigma^2 * sqrt(2/l)": SIGMA2 * np.sqrt(2 / l)})
display(pd.DataFrame(rows).set_index("l").round(4))

> **Вывод.** Смещения нет ни при каком $\ell$ — а что происходит с разбросом $Q$ при росте $\ell$ вчетверо?
>
> *(ваш ответ здесь)*

### А теперь алгоритм подбирается по той же выборке

Ровно то, что предписывает принцип ERM: берём выборку, обучаем на ней МНК,
и на ней же считаем $Q$. Формально утверждение 1.14 больше не применимо —
посмотрим, что будет.

In [ ]:
p = 2                                        # число параметров модели
Q_train, R_hat = np.empty(N_REPEAT), np.empty(N_REPEAT)
for t in range(N_REPEAT):
    X, y = sample(ELL, gen)
    theta = np.linalg.lstsq(X, y, rcond=None)[0]
    Q_train[t] = empirical_risk(loss_squared, X @ theta, y)
    R_hat[t] = empirical_risk(loss_squared, X_big @ theta, y_big)

print(f"sigma^2                    = {SIGMA2:.5f}")
print(f"среднее Q(a_hat, X^l)      = {Q_train.mean():.5f}"
      f"   теория sigma^2(1 - p/l) = {SIGMA2 * (1 - p / ELL):.5f}")
print(f"среднее R(a_hat)           = {R_hat.mean():.5f}"
      f"   теория sigma^2(1 + p/l) = {SIGMA2 * (1 + p / ELL):.5f}")
print(f"зазор R - Q                = {(R_hat - Q_train).mean():.5f}"
      f"   теория 2 sigma^2 p/l    = {2 * SIGMA2 * p / ELL:.5f}")

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(0, np.percentile(R_hat, 99.5), 60)
ax.hist(Q_train, bins=bins, alpha=0.65, density=True, label=r"$Q$ на обучающей")
ax.hist(R_hat, bins=bins, alpha=0.65, density=True, label=r"$R$ истинный")
ax.axvline(SIGMA2, color="black", lw=2, label=r"$\sigma^2$")
ax.set_xlabel("риск"); ax.set_ylabel("плотность"); ax.legend()
ax.set_title("Алгоритм выбран по выборке: обучающая ошибка смещена вниз")
plt.tight_layout(); plt.show()

> **Вывод.** Для фиксированного алгоритма смещения нет, для обученного — есть. В каком месте доказательства утверждения 1.14 ломается рассуждение? Как зазор зависит от $\ell$ и числа параметров $p$?
>
> *(ваш ответ здесь)*

---
# Часть 4. Полиномы: первое переобучение

Как отмечено в примере 1.8 конспекта, признаками линейной модели могут служить
любые функции исходных данных: при $f_j(x) = x^{\,j-1}$ та же формула задаёт
полином. Модель остаётся **линейной по параметрам**, весь аппарат МНК применим.

Возьмём всего 15 точек — чтобы эффект был виден сразу — и посмотрим, что
происходит с ростом степени.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

true_f = lambda t: np.sin(3 * t) + 0.5 * t
x_tr = np.sort(rng.uniform(-1, 1, 15))          # всего 15 наблюдений
x_te = np.sort(rng.uniform(-1, 1, 500))
y_tr = true_f(x_tr) + rng.normal(0, 0.25, 15)
y_te = true_f(x_te) + rng.normal(0, 0.25, 500)

fit_poly = lambda d: make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(
    x_tr[:, None], y_tr)

### Задание 4.1. Ошибка на обучении и на контроле

Для каждой степени посчитайте эмпирический риск на обучающей выборке и на
контрольной. Обратите внимание: контрольная выборка в обучении не участвовала.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

degrees = [1, 2, 3, 5, 7, 9, 11]
rows = []
for d in degrees:
    model = fit_poly(d)
    # TODO (2 строки): Q на обучающей (x_tr, y_tr) и на контрольной (x_te, y_te)
    rows.append({"степень": d, "Q обучающая": ..., "Q контрольная": ...})
table = pd.DataFrame(rows).set_index("степень")
display(table.round(4))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
grid = np.linspace(-1, 1, 400)
ax1.scatter(x_tr, y_tr, s=30, color="black", zorder=3, label="обучающая выборка")
ax1.plot(grid, true_f(grid), "k--", lw=1.5, label="истинная зависимость")
for d in (1, 3, 9):
    ax1.plot(grid, fit_poly(d).predict(grid[:, None]), lw=1.8, label=f"степень {d}")
ax1.set_ylim(y_tr.min() - 1, y_tr.max() + 1); ax1.legend(fontsize=8)
ax1.set_xlabel("$x$"); ax1.set_ylabel("$y$"); ax1.set_title("Полиномы разных степеней")

ax2.plot(table.index, table["Q обучающая"], "o-", lw=2, label="обучающая")
ax2.plot(table.index, table["Q контрольная"], "s-", lw=2, label="контрольная")
ax2.set_yscale("log"); ax2.set_xlabel("степень"); ax2.set_ylabel("$Q$")
ax2.set_title("Риск против сложности модели"); ax2.legend()
plt.tight_layout(); plt.show()

> **Вывод.** Как ведут себя обе ошибки с ростом степени? Какая степень оптимальна и можно ли было выбрать её по обучающей ошибке?
>
> *(ваш ответ здесь)*

---
# Часть 5. Чем измерять качество регрессии

Лекция измеряет качество эмпирическим риском $Q$. На практике этого мало:
$Q$ выражен в квадратах единиц измерения ответа и ни с чем не сравним.
Введём рабочий набор метрик — он понадобится во всех остальных занятиях.

$$
\mathrm{MSE} = \frac1\ell\sum_i (a(x_i) - y_i)^2, \qquad
\mathrm{RMSE} = \sqrt{\mathrm{MSE}}, \qquad
\mathrm{MAE} = \frac1\ell\sum_i |a(x_i) - y_i|,
$$

$$
R^2 = 1 - \frac{\sum_i (a(x_i) - y_i)^2}{\sum_i (\overline y - y_i)^2}
= 1 - \frac{\mathrm{MSE}(a)}{\mathrm{MSE}(\text{константа }\overline y)} .
$$

MSE и MAE — это ровно $Q$ при квадратичной и абсолютной потере, то есть ничего
нового. А вот $R^2$ устроен иначе: он сравнивает модель с константным прогнозом,
поэтому **безразмерен** и сопоставим между задачами. $R^2 = 0$ — качество
константы, $R^2 < 0$ — модель хуже константы.

### Задание 5.1. Четыре метрики своими руками

Библиотечные метрики появятся в следующей ячейке — для сверки. Сначала свои.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def mse(y_true, y_pred): ...
def rmse(y_true, y_pred): ...
def mae(y_true, y_pred): ...


def r2(y_true, y_pred):
    """1 - MSE(модель) / MSE(константа = среднее y_true)."""
    # TODO (4 короткие функции). R^2 выразите через уже написанный mse.
    raise NotImplementedError

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)
const = np.full_like(y_test, y_train.mean())

display(pd.DataFrame({
    "линейная регрессия": {"MSE": mse(y_test, pred), "RMSE": rmse(y_test, pred),
                           "MAE": mae(y_test, pred), "R^2": r2(y_test, pred)},
    "константа (среднее)": {"MSE": mse(y_test, const), "RMSE": rmse(y_test, const),
                            "MAE": mae(y_test, const), "R^2": r2(y_test, const)},
}).T.round(4))

print(f"сверка со sklearn.metrics: MSE {abs(mse(y_test, pred) - mean_squared_error(y_test, pred)):.1e}, "
      f"MAE {abs(mae(y_test, pred) - mean_absolute_error(y_test, pred)):.1e}, "
      f"R^2 {abs(r2(y_test, pred) - r2_score(y_test, pred)):.1e}")
print(f"целевая переменная: от {y.min():.0f} до {y.max():.0f}, "
      f"среднее {y.mean():.1f}, ст. откл. {y.std():.1f}")

> **Вывод.** MSE вышла около 2850. Это много или мало? Почему по $R^2$ на этот вопрос ответить можно, а по MSE — нет? И почему $R^2$ константы оказался слегка отрицательным, а не нулевым?
>
> *(ваш ответ здесь)*

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
lims = [y_test.min() - 20, y_test.max() + 20]
ax1.scatter(y_test, pred, s=22, alpha=0.65)
ax1.plot(lims, lims, "k--", lw=1.5, label="идеальный прогноз")
ax1.set_xlim(lims); ax1.set_ylim(lims); ax1.legend(fontsize=8)
ax1.set_xlabel("истинное $y$"); ax1.set_ylabel("предсказание $a(x)$")
ax1.set_title("Предсказание против истины")

resid = y_test - pred
ax2.hist(resid, bins=25, density=True, alpha=0.7)
ax2.axvline(0, color="black", lw=1.5)
ax2.set_xlabel("остаток $y - a(x)$"); ax2.set_ylabel("плотность")
ax2.set_title(f"Остатки: среднее {resid.mean():.1f}, ст. откл. {resid.std():.1f}")
plt.tight_layout(); plt.show()

> **Вывод.** Что видно на диаграмме «предсказание против истины» и о чём говорит форма распределения остатков?
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Утверждение 1.14 говорит, что $\mathbb{E}[Q] = R$. Почему это не противоречит переобучению? Сформулируйте условие, при котором равенство верно.
2. Вы получили $Q(\hat a, X^\ell) = 0$. Что это говорит о качестве на новых объектах? Приведите алгоритм с нулевым эмпирическим риском и максимально плохим истинным.
3. Почему $\theta^* = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y$ — правильная формула, но неправильная реализация?
4. Модель полиномов степени $\le d$ вложена в модель степени $\le d+1$. Докажите, что минимум эмпирического риска по большей модели не больше. Почему из этого **не** следует, что большая модель лучше?
5. Две модели на разных задачах дали MSE 12 и MSE 3400. Какая работает лучше? Какого числа не хватает, чтобы ответить?

---

**Дома:** откройте `lab02_homework.ipynb` — там три задачи на вашей собственной таблице: матричное дифференцирование, пять способов решить МНК и связь функции потерь с распределением шума.